# TRACT: From Zero-Shot to Expert-Reviewed Security Crosswalk

**A practitioner's guide to mapping AI security frameworks using machine learning**

This notebook tells the story of how we built a system that reads any security control and tells you which part of a universal security taxonomy it belongs to. We'll walk through every experiment, every failure, and every hard-won insight — in plain language.

---

In [ ]:
"""Setup: imports, seeds, prerequisite checks."""
import sys
import json
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from sklearn.manifold import TSNE

# Ensure nb_helpers is importable from notebooks/
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent))

from nb_helpers import (
    PROJECT_ROOT, RESULTS_DIR, DATA_DIR,
    PHASE0_DIR, PHASE1B_DIR, PHASE1C_DIR,
    REVIEW_DIR, BRIDGE_DIR, DATASET_DIR,
    OKABE_ITO, SEQUENTIAL_BLUE, SEQUENTIAL_ORANGE, DIVERGING,
    FigureCounter, style_axes, plotly_with_fallback,
    load_phase0_experiment, load_firewalled_baseline,
    load_fold_metrics, load_training_logs,
    load_calibration_data, load_deployment_embeddings,
    load_review_metrics, load_review_export,
    load_cre_hierarchy, load_crosswalk, load_framework_metadata,
    check_prerequisites,
)

# Reproducibility
random.seed(42)
np.random.seed(42)

# Matplotlib defaults
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "savefig.bbox": "tight",
})
sns.set_style("whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

# Figure counter — instantiate once
fig_counter = FigureCounter()

# Prerequisite check
missing = check_prerequisites()
if missing:
    print("⚠️  Missing prerequisite files:")
    for m in missing:
        print(f"   - {m}")
    print("\nSee the notebook README for setup instructions.")
else:
    print("✓ All prerequisite files found")

# Verify CWD
cwd = Path.cwd()
if cwd.name != "notebooks":
    print(f"⚠️  Expected CWD = notebooks/, got {cwd.name}/. Shell cells may not work correctly.")
else:
    print(f"✓ CWD: {cwd}")

print(f"✓ PROJECT_ROOT: {PROJECT_ROOT}")
print(f"✓ NumPy {np.__version__}, Matplotlib {matplotlib.__version__}")